# Titel der Analyse

Untertitel

Dein Name

## LLM-Setup

Konfiguriere deinen AI-Helfer einmalig vor Beginn deiner Analyse. Führe die Code-Zellen in diesem Abschnitt bei jeder Verwendung dieses Notebooks einmalig aus.

Wenn etwas schief geht, kannst du die folgenden Zellen nochmals ausführen, um in den Startzustand zurückzukehren.

### Ollama Rechenwerk

In [1]:
import ollama

Entscheide dich, ob du **entweder** Ollama-Cloud **oder** einen lokalen Ollama-Server verwenden willst. Wo läuft dein Modell?

#### Cloud

Dein Computer ist zu langsam oder hat nicht genügend RAM (16GB+)? Dann verwende Ollama Cloud. Informationen zu Ollama-Cloud findest du unter [docs.ollama.com/cloud](https://docs.ollama.com/cloud#cloud-models)

In [2]:
OLLAMA_API_KEY="replace-me-with-a-valid-key"

Deinen eigenen privaten API-Schlüssel erstellst du nach dem Login unter [ollama.com/settings/keys](https://ollama.com/settings/keys).

In [3]:
client = ollama.Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

#### Local

Alternativ kannst du Ollama auch lokal laufen lassen. **Führe die folgende Zellen nur aus, wenn du Ollama Cloud in diesem Notebook NICHT verwendest.**

In [ ]:
client = ollama.Client(
#  host='http://127.0.0.1:11434' # Ollama und Juypter laufen lokal
  host='http://10.0.2.2:11434' # Ollama äuft lokal und Jupyter in der VM
)

Ob unter einer bestimmten (lokalen) IP-Adresse und Port ein (lokaler) Ollama-Server läuft, kannst du leicht so testen:

In [ ]:
%%bash
#curl --silent http://127.0.0.1:11434
curl --silent http://10.0.2.2:11434

### Hilfsfunktionen

Für die einfachere Handhabung nutzen wir die zwei selbstgeschriebenen Funktionen `ask()` und `chat()`. Unser Code im Notebook ist dann übersichtlicher.

In [4]:
from IPython.display import display, Markdown
import pandas as pd
import json
import os

def load_conversation_log(filename = "llm_conversation_log.json"):
    """Liest ein .json-Datei, welche eine Aufzeichnung einer Unterhaltung enthält, ein."""
    global messages
    if os.path.exists(filename):
        with open(filename, 'r') as file:
            content = json.loads(file.read())
        if content[0].get("role") == "system":
            print("Loading", filename)
            messages = content

def save_conversation_log(filename = "llm_conversation_log.json"):
    """Schreibt die aktuelle Aufzeichnung der Unterhaltung in eine .json-Datei."""
    with open( filename , "w" ) as file:
        json.dump( messages , file )
    
def ask(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell - ohne Historie"""
    global client
    global model_name
    response = client.generate(model=model_name, prompt=prompt)
    display(Markdown(response.response))

def remove_message(msg_list, n=1):
    """Entfernt die ältesten n einträge aus einer Unterhaltung"""
    #print("Cleanup - removing {} oldest message from chat history".format(n))
    return msg_list[0:1] + msg_list[(n+1):]
    
def chat(prompt):
    """Einfacher Chat-Bot mit global spezifiziertem Client, Modell und Historie"""
    global client
    global model_name
    global messages
    global max_chat_history
    
    messages.append({"role": "user", "content": prompt})

    if max_chat_history != 0:
        message_count = len(messages) - 1
        
        # remove exess chat history, if we have fixed size max_chat_history
        excess_messages = message_count - max_chat_history
        if max_chat_history > 0 and excess_messages > 0:
            messages = remove_message(messages, excess_messages)

        # if context window is (still) to long, we always remove old chat entries
        for _ in range(message_count + 1):
            try:
                response = client.chat(model=model_name, messages=messages)
            except ollama.ResponseError as e:
                if len(messages) > 1:
                    messages = remove_message(messages, 1)
                    continue
                raise
            except Exception as e:
                raise
    else:
        # no context lenght limit - expect your client to die eventually
        response = client.chat(model=model_name, messages=messages)
        
    messages.append({"role": "assistant", "content": response.message.content})
    save_conversation_log()
    display(Markdown(response.message.content))

def show_models_from_(output):
    """Gibt die Modelle von client.list() als Pandas Dataframe zurück"""
    df = pd.DataFrame([
        {
            "Model": entry.get("model", ""),
            "Size [GB]": round(int(entry.get("size", 0)) / 1024**3, 1)
        }
        for entry in output.get("models", [])
    ])

    if not df.empty:
        df = df.sort_values(by="Model", key=lambda s: s.str.lower())

    df = df.reset_index(drop=True)
    return df

Wichtig: Die globalen Variablen `model_name` und `messages` und `max_chat_history` müssen vor der Verwendung definiert werden. Dazu alle Code-Zellen bis **Modelltest** einmalig nacheinander ausführen. 

### Chat-History


Die Funktion `chat()` verwendet auch die bisherige Konversation als Eingabe - so "erinnert" sich ein Modell, was alles schon besprochen wurde. Die maximal Länge der Eingabe ist von Modell zu Modell verschieden aber immer begrenzt - genau wie die maximale Länge der Ausgabe eines Modells. Mit der globalen Variable `max_chat_history` kannst du einstellen, wie viele der vorhergehenden Fragen und Antworten im `chat()` verwendet werden.

|Wert|Bedeutung|
|--|--|
|-1|Model Limit - das heisst, dass dein Chat-Client soviel History wie möglich verwendet. Wenn du einen langsamen Computer hast, ist diese Einstellung keine gute Idee. Wenn du Ollama-Cloud verwendest oder einen sehr schnellen Computer hast, dann ist ggf. -1 eine gute Wahl|
|0|ohne Limit - das heisst, dass dein Chat-Client irgendwann abstützen wird - nur zu empfehlen zum Testen des "maximum context window" - also der maximalen Eingabelänge eines Modells.|
|3,4,5,...|festes Limit - Wenn du bspw. 3 wählst, dann "erinnert" sich dein Chatbot maximal an die drei vorhergehenden Fragen und Antworten. Wenn Ollama lokal läuft und/oder dein Computer langsam ist, solltest du eine kleine Zahl wie bspw. `5` wählen. Versuch macht kluch!|

In [5]:
max_chat_history = 7

### Modellauswahl

Modellübersicht: https://ollama.com/search

Welche Modelle gibt es auf deinem Ollama-Server oder bei Ollama-Cloud?

In [6]:
show_models_from_(client.list())

,Model,Size [GB]
0,cogito-2.1:671b,641.3
1,deepseek-v3.1:671b,641.3
2,deepseek-v3.2,641.3
3,devstral-2:123b,119.4
4,devstral-small-2:24b,48.1
5,gemini-3-flash-preview,0.0
6,gemma3:12b,22.4
7,gemma3:27b,51.2
8,gemma3:4b,8.0
9,glm-4.6,648.3


Wenn Ollama lokal läuft, achte auf die Grösse des Modells (in GB) im Verhältnis zum verfügbaren Arbeitsspeicher. Grösser heisst i.d.R. auch langsamer - aber nicht zwangsläufig auch immer besser.

Mit welchem Modell willst du arbeiten?

In [7]:
model_name = 'gemma3:4b' # Starte mit diesem, wenn du Ollama-Cloud verwendest

#model_name = 'gemma3:270m' # Starte mit diesem, wenn du Ollama lokal laufen lässt
#model_name = 'gemma3:1b'

# weitere Beispiele
#model_name = 'deepseek-r1:1.5b'
#model_name = 'codegemma:2b'

Wenn du **nicht** Ollama-Cloud verwendest, installierst du neue Modelle auf dem Ollama-Server, mit dem du gerade verbunden bist, so: (entferne in dem Fall die #)

In [8]:
#client.pull(model_name)

oder so:

In [9]:
#%%bash
#ollama pull gemma3:270m

### Priming

Entweder starten wir ganz frisch **ODER** wir setzen eine Unterhaltung fort.

#### Frischer Start

Wir speichern die gesamte Unterhaltung in der globalen Variablen `messages`. Am Anfang können wir das Modell noch `primen`.

In [10]:
priming = """
You are the best Python coder in the world.
You know Pandas and all of Pandas functionality inside out.
You use directly Pandas data frames for plotting and do NOT use matplotlib or seaborn explicitly.
You code like a student in 10th grade and prefer simple solutions.
You give short and informative answers.
"""

In [11]:
messages = [
    {"role": "system", "content": priming}
]

#### Unterhaltung fortsetzen

Falls vorhanden, wird die Datei `llm_conversation_log.json` in die globale Variable `messages` eingelesen.

In [12]:
load_conversation_log()

Loading llm_conversation_log.json


### Modelltest

Wir testen, ob alle Variablen und Funktionen vorhanden sind - falls nicht, hast du vermutlich vergessen, eine der vorhergehenden Code-Zellen auszuführen.

In [13]:
MUST_HAVE_OBJECTS = ('ollama', 'client', 'chat', 'ask', 'max_chat_history', 'model_name', 'priming', 'messages')

llm_setup_errors = 0
for one_object in MUST_HAVE_OBJECTS:
    if one_object not in locals():
        llm_setup_errors += 1
        print("ERROR:", one_object, "nicht gefunden - bitte die zugehörige, vorhergehende Code-Zelle einmalig ausführen!")
        
assert llm_setup_errors == 0

Ausserdem testen wir, ob du eine Modell ausgewählt hast, was auch auf dem Ollama-Server vorhanden ist.

In [14]:
assert model_name in list(show_models_from_(client.list())["Model"])

### Beispiele

Für einmalige Fragen nutzen wir `ask()`

In [15]:
ask("Erzähl einen besonders lustigen Informatik-Lehrerwitz!")

Warum sind Computer so schlecht im Golf?

… Weil sie immer nur „Bugs“ machen! 😄


Für Unterhaltungen nutzen wir `chat()` - mit `"""` auch mehrzeilig.

In [16]:
chat("""
Ich benötige Hilfe bei der Datenanlyse mit Python und Pandas.
Wie gehe ich von den Rohdaten, die ich als .csv-Datei habe, bis zur Visualisierung meiner Ergebnisse Schritt für Schritt vor?
Ich kenne die Pandas-Funktionen .describe() .pivot_table() .groupby() und .corr().
Ich möchte beispielsweise Bar- und Lineplots, Scatterplots, Boxplots, Violinplots und Heatmaps erzeugen.
""")

Okay, lass uns das aufschlüsseln! Hier ist der Prozess mit Beispielen:

1.  **Daten laden:** `df = pd.read_csv('deine_datei.csv')`
2.  **Daten verstehen:** `df.head()`, `df.info()`, `df.describe()` (wie vorher)
3.  **Daten vorbereiten:** Bereinigen (Fehlerhafte Werte, fehlende Daten)
4.  **Analyse:**  Verwende `.groupby()`, `.pivot_table()`, `.corr()` wie du kennst.
5.  **Visualisierung:**  Verwende `df.plot()` für einfache Plots.

**Beispiele für Visualisierungen:**

*   **Barplot:** `df['Spalte'].value_counts().plot(kind='bar')` (zeigt die Häufigkeit von Werten in einer Spalte)
*   **Lineplot:** `df.plot(x='Datum', y='Wert', kind='line')`
*   **Scatterplot:** `df.plot(x='Spalte1', y='Spalte2', kind='scatter')`
*   **Boxplot:** `df.boxplot(column='Spalte')`
*   **Violinplot:** `df.plot(kind='violin', column='Spalte')`
*   **Heatmap (Korrelation):** `df.corr().plot(kind='heatmap')`

**Wichtig:**  `kind` ist der Parameter, der bestimmt, welchen Plot du erzeugst.

Welche Art von Daten hast du und was möchtest du visualisieren?  Dann kann ich dir spezifischere Hinweise geben.

## Forschungsfragen

1. ...
2. ...
3. ...

## Daten einlesen

In [17]:
chat("""
Wie liest man eine komma-separierte Datei in Python mit Pandas ein, 
wenn das Trennzeichen ein Semikolon ist? Was ändert sich, wenn das Trennzeichen ein Komma ist?
""")

Das ist einfach!

*   **Semikolon als Trennzeichen:**
    ```python
    import pandas as pd
    df = pd.read_csv('deine_datei.csv', sep=';')
    ```
    Hier sagen wir Pandas, dass das Semikolon das Trennzeichen ist.

*   **Komma als Trennzeichen:**
    ```python
    import pandas as pd
    df = pd.read_csv('deine_datei.csv', sep=',')
    ```
    Hier sagen wir Pandas, dass das Komma das Trennzeichen ist.

Der einzige Unterschied ist, welches Zeichen du als Trennzeichen angibst.  Pandas geht davon aus, dass die Werte in jeder Zeile durch das angegebene Trennzeichen getrennt sind.


In [18]:
import pandas as pd

In [19]:
df = pd.read_csv('KSZO-solar-data.csv', sep=",")
df

,time_utc,time_local,consumption,production,battery_charging,battery_discharging,battery_currentPower,battery_activeDevice,smart_meter_currentPower,smart_meter_activeDevice,inverter_WR_MS1_currentPower,inverter_WR_MS1_activeDevice,inverter_SolarEdge_SE_currentPower,inverter_SolarEdge_SE_activeDevice,inverter_WR_MS2_currentPower,inverter_WR_MS2_activeDevice
0,2023-12-28T15:45:00,2023-12-28T16:45:00,41640,0,0,0,52.00,1.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0
1,2023-12-28T16:00:00,2023-12-28T17:00:00,42226,0,0,0,52.00,1.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0
2,2023-12-28T16:15:00,2023-12-28T17:15:00,42226,0,0,0,52.00,1.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0
3,2023-12-28T16:30:00,2023-12-28T17:30:00,42226,0,0,0,52.00,1.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0
4,2023-12-28T16:45:00,2023-12-28T17:45:00,42226,0,0,0,52.00,1.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77163,2026-03-11T11:30:00,2026-03-11T12:30:00,66987,90984,0,0,-10019.00,1.0,-23996.33,1.0,30900.67,1.0,30900.67,1.0,29182.33,1.0
77164,2026-03-11T11:45:00,2026-03-11T12:45:00,62299,87617,0,0,-10026.00,1.0,-25317.33,1.0,29737.33,1.0,29737.33,1.0,28143.00,1.0
77165,2026-03-11T12:00:00,2026-03-11T13:00:00,60324,136516,0,0,-10012.33,1.0,-76339.00,1.0,46576.00,1.0,46576.00,1.0,43364.33,1.0
77166,2026-03-11T12:15:00,2026-03-11T13:15:00,65548,106002,0,0,-10008.00,1.0,-41052.00,1.0,36034.67,1.0,36034.67,1.0,33933.00,1.0


In [20]:
df.describe()

,consumption,production,battery_charging,battery_discharging,battery_currentPower,battery_activeDevice,smart_meter_currentPower,smart_meter_activeDevice,inverter_WR_MS1_currentPower,inverter_WR_MS1_activeDevice,inverter_SolarEdge_SE_currentPower,inverter_SolarEdge_SE_activeDevice,inverter_WR_MS2_currentPower,inverter_WR_MS2_activeDevice
count,77168.000000,77168.000000,77168.0,77168.0,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000,75459.000000
mean,26225.697323,29743.150853,0.0,0.0,397.163727,0.773453,-2097.168527,0.783790,10526.983801,0.445951,9775.468057,0.395579,10116.253182,0.446203
std,21170.904711,57284.240071,0.0,0.0,3277.503354,0.418600,42069.929279,0.411662,19685.964894,0.497073,19514.176152,0.488978,19206.374419,0.497101
min,0.000000,0.000000,0.0,0.0,-10057.670000,0.000000,-245751.330000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9524.000000,0.000000,0.0,0.0,-52.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,16702.000000,0.000000,0.0,0.0,-52.000000,1.000000,8467.670000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,42226.000000,28662.250000,0.0,0.0,0.000000,1.000000,10705.165000,1.000000,10810.625000,1.000000,8179.395000,1.000000,9936.325000,1.000000
max,109785.000000,268980.000000,0.0,0.0,19793.000000,1.000000,84918.810000,1.000000,89854.560000,1.000000,89972.220000,1.000000,89283.110000,1.000000


## Daten vorverarbeiten

## Daten analysieren

## Daten visualisieren